<h2>Import Libraries<h2>

In [3314]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import RandomOverSampler


<h2>Import Dataset<h2>

In [3315]:
df_loan=pd.read_csv('../data/cleaned_data/cleaned_loan_dataset.csv')
df_loan

,loan_id,gender_male,married_yes,dependents_1.0,dependents_2.0,dependents_3.0,self_employed_yes,credit_history_1.0,loan_amount,loan_amount_term,applicant_income,coapplicant_income,loan_status,total_income,estimated_monthly_installment,balance_income
0,LP001002,1,0,0,0,0,0,1,128000.0,360,5849,0.0,1,5849.0,355.6,5493.4
1,LP001003,1,1,1,0,0,0,1,128000.0,360,4583,1508.0,0,6091.0,355.6,5735.4
2,LP001005,1,1,0,0,0,1,1,66000.0,360,3000,0.0,1,3000.0,183.3,2816.7
3,LP001006,1,1,0,0,0,0,1,120000.0,360,2583,2358.0,1,4941.0,333.3,4607.7
4,LP001008,1,0,0,0,0,0,1,141000.0,360,6000,0.0,1,6000.0,391.7,5608.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,0,0,0,0,0,0,1,71000.0,360,2900,0.0,1,2900.0,197.2,2702.8
610,LP002979,1,1,0,0,1,0,1,40000.0,180,4106,0.0,1,4106.0,222.2,3883.8
611,LP002983,1,1,1,0,0,0,1,253000.0,360,8072,240.0,1,8312.0,702.8,7609.2
612,LP002984,1,1,0,1,0,0,1,187000.0,360,7583,0.0,1,7583.0,519.4,7063.6


<h2>Separate Features and Target</h2>

<p>Drop the loan_id (identifier, not a predictive feature) and isolate the target variable loan_status.</p>

In [3316]:
X = df_loan.drop(columns=['loan_id', 'loan_status',  'total_income', 'balance_income'])
y = df_loan['loan_status']

In [3317]:
print('Features shape:', X.shape)
print('Target shape:', y.shape)

Features shape: (614, 12)
Target shape: (614,)


<h2>Split into Train and Test Sets</h2>

<p>Use a stratified split to preserve the loan_status class distribution in both sets. The split happens before any scaling/encoding is fit
to avoid data leakage.</p>

In [3318]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42)

In [3319]:
print('Train features shape:', X_train.shape)
print('Test features shape :', X_test.shape)

Train features shape: (491, 12)
Test features shape : (123, 12)


<h2>Scale Numeric Features</h2>
<p>Scale only the numeric columns using StandardScaler. Fit the scaler on the training set only (to avoid data leakage) and apply it to both train and test. </p>

In [3320]:
# Separate numeric and categorical/encoded columns
numeric_cols = ['loan_amount', 'loan_amount_term', 'applicant_income',
                'coapplicant_income',
                'estimated_monthly_installment']
    
cat_cols = [c for c in X.columns if c not in numeric_cols]

print('Numeric columns:', numeric_cols)
print('Categorical (unscaled) columns:', cat_cols)

Numeric columns: ['loan_amount', 'loan_amount_term', 'applicant_income', 'coapplicant_income', 'estimated_monthly_installment']
Categorical (unscaled) columns: ['gender_male', 'married_yes', 'dependents_1.0', 'dependents_2.0', 'dependents_3.0', 'self_employed_yes', 'credit_history_1.0']


In [3321]:
# Scale only the numeric columns and fit on train, apply to both
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

<h3>Feature Selection</h3>

In [3322]:
# Five most statistically relevant features using training data only
selector = SelectKBest(score_func=f_classif,  k=5)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X_train_scaled.columns[selector.get_support()]

print("Selected features:")
print(list(selected_features))
print("Training shape:", X_train_selected.shape)
print("Testing shape :", X_test_selected.shape)

Selected features:
['married_yes', 'dependents_1.0', 'dependents_2.0', 'credit_history_1.0', 'coapplicant_income']
Training shape: (491, 5)
Testing shape : (123, 5)


<h3>Helper function to evaluate the train and test accuracy</h3>

In [3323]:
# A helper function to evaluate the train and test accuracy
def evaluate_accuracy(model, X_train_selected, y_train, X_test_selected, y_test):

    # Train accuracy
    y_pred_train = model.predict(X_train_selected)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    
    # Test accuracy
    y_pred_test = model.predict(X_test_selected)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    print(f'Train Accuracy: {round((train_accuracy * 100), 2)}%')
    print(f'Test Accuracy: {round((test_accuracy * 100) ,2)}%')

    print(" ")

    # Classification Report
    print("Classification Report")
    print(classification_report(y_test, y_pred_test))

    # Confusion Matrix
    print("Confusion Matrix")
    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred_test),
        columns=['Predicted N', 'Predicted Y'], 
        index=['Actual N', 'Actual Y']
    )
    print(cm)


<h3>Logistic Regression Model</h3>

In [3324]:
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_selected, y_train)

In [3325]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_ros, y_train_ros)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

<h3>Random Forest Model</h3>

In [3326]:
rf_model = RandomForestClassifier(max_depth = 5, class_weight= 'balanced', n_estimators= 50, random_state=42)
rf_model.fit(X_train_selected, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 Th

<h3>Evaluate Model Performance</h3>

In [3327]:
# Logistic Regression Model Performance
evaluate_accuracy(lr_model, X_train_selected, y_train, X_test_selected, y_test)

Train Accuracy: 79.43%
Test Accuracy: 85.37%
 
Classification Report
              precision    recall  f1-score   support

           0       0.86      0.63      0.73        38
           1       0.85      0.95      0.90        85

    accuracy                           0.85       123
   macro avg       0.85      0.79      0.81       123
weighted avg       0.85      0.85      0.85       123

Confusion Matrix
          Predicted N  Predicted Y
Actual N           24           14
Actual Y            4           81


In [3328]:
# Random Forest Model Performance
evaluate_accuracy(rf_model, X_train_selected, y_train, X_test_selected, y_test)

Train Accuracy: 80.86%
Test Accuracy: 83.74%
 
Classification Report
              precision    recall  f1-score   support

           0       0.78      0.66      0.71        38
           1       0.86      0.92      0.89        85

    accuracy                           0.84       123
   macro avg       0.82      0.79      0.80       123
weighted avg       0.83      0.84      0.83       123

Confusion Matrix
          Predicted N  Predicted Y
Actual N           25           13
Actual Y            7           78
